# Intersection Control: Signals vs. Scheduling

Run the microsimulation yourself, in the browser. Nothing to install.

This is the study from [ruud0/intersection-control-sim](https://github.com/ruud0/intersection-control-sim):
a time-stepped simulation of a six-signal urban grid, written from scratch in Python with no
simulation libraries, with no SUMO and no SimPy. Three control strategies are compared on the same
network and the same seeded demand:

| Strategy | How it decides |
|---|---|
| **Fixed-time** | Webster's formula: cycle length from critical flow ratios, green split proportional |
| **Semi-actuated** | Arterial rests in green; side street served on call, with gap-out and max-out |
| **Reservation scheduling** | No phases. Vehicles book a time slot at the intersection and get a speed advisory to land on it |

**Run all cells** (`Runtime → Run all`, or ⌘/Ctrl+F9). The whole notebook takes about 5 minutes.


## 1. Get the code


In [ ]:
!git clone --quiet --depth 1 https://github.com/ruud0/intersection-control-sim.git
%cd intersection-control-sim
!pip install --quiet -r requirements.txt
print('ready')


## 2. Run the experiment

`--quick` runs a reduced sweep of 2 demand levels × 2 seeds × 3 controllers, so it finishes in
a couple of minutes on Colab's 2 cores. Drop the flag for the full sweep (5 levels × 8 seeds,
120 runs, 240k+ vehicles), which is what the numbers in the README come from.

Every cell of the design sees an identical network and identically seeded demand, so the only
thing varying within a cell is the control strategy.


In [ ]:
!python run.py --quick


## 3. The results table


In [ ]:
import csv
from collections import defaultdict

rows = list(csv.DictReader(open('results/results.csv')))

# Average the replications within each (controller, demand) cell.
cells = defaultdict(list)
for r in rows:
    cells[(r['controller'], int(r['demand']))].append(r)

cols = [('stops_per_vehicle', 'Stops/veh'), ('mean_delay', 'Mean delay (s)'),
        ('p95_delay', 'p95 delay (s)'), ('ped_mean_wait', 'Ped wait (s)')]

hdr = f"{'Controller':<24}{'Demand':>8}" + ''.join(f'{lbl:>16}' for _, lbl in cols)
print(hdr)
print('-' * len(hdr))
for (ctrl, dem) in sorted(cells, key=lambda t: (t[1], t[0])):
    group = cells[(ctrl, dem)]
    line = f'{ctrl:<24}{dem:>8}'
    for k, _ in cols:
        line += f'{sum(float(g[k]) for g in group) / len(group):>16.1f}'
    print(line)


## 4. The figures

The same charts the README leads with, regenerated from the run you just did.
Error bars are 95% confidence intervals across seeds, and with `--quick` there are only two seeds
per cell, so expect them to be wide.


In [ ]:
!python analyze.py

from IPython.display import Image, display
display(Image('docs/headline.png'))
display(Image('docs/stops_vs_demand.png'))


## 5. Watch it run

Vehicles are dots on their links, coloured by whether they are moving or stopped, so the
stop-and-go difference between strategies is visible directly. Change `controller` below and
re-run the cell to compare: `fixed` platoons hard at the stop bars, while `scheduler` keeps them
rolling.

Rendering takes about a minute.


In [ ]:
controller = 'scheduler'  #@param ['fixed', 'actuated', 'scheduler']
demand     = 2000         #@param {type:'slider', min:1000, max:3000, step:500}
seconds    = 60           #@param {type:'slider', min:20, max:120, step:20}

!python animate.py --controller {controller} --demand {demand} --seconds {seconds} --out docs/run.gif

from IPython.display import Image, display
display(Image('docs/run.gif'))


## What the study found

Reservation scheduling cuts **stops per vehicle by 4.5–5.1×** against a Webster-timed signal,
at every demand level tested. That part holds up. Two things complicate it:

**The delay advantage inverts under load.** Below roughly 1,800 veh/h scheduling has the lowest
delay of the three. Above it, semi-actuated signals take over and stay ahead: 90.9 s vs 106.5 s
at 2,000 veh/h, 92.1 s vs 127.7 s at 3,000.

**Scheduling pays for its smoothness at the crosswalk.** Mean pedestrian wait under scheduling
climbs from 14.3 s to 34.6 s as demand rises (a 2.4× increase) while both signal strategies
hold flat near 13 s. Continuous vehicle flow means there is never a natural gap to release a
pedestrian into, so the crossing has to be forced, and under load it gets forced late.

Pedestrians are modelled as first-class agents with MUTCD-compliant timing: 4 s WALK plus a
clearance interval computed at the 3.5 ft/s design walking speed.

---

Full write-up, method and figures: **[github.com/ruud0/intersection-control-sim](https://github.com/ruud0/intersection-control-sim)**
